# NLP Assignment 2025/26 - PoliMillionaire Starter

**Goal.** Build and evaluate local chatbot models for the online quiz *Who wants to be a PoliMillionaire?* using the official `millionaire_client` package.

**Group members.** TODO: add names and Polimi email addresses.

**Video link.** TODO: add the screen-capture video link.

**Coding assistants statement.** TODO: state which coding assistants, if any, were used.

**Current strategy.** Text-only game mode, two local open-weight models, optional ensemble voting, no RAG in the first implementation.

## 1. Colab and project path

This cell mounts Google Drive and adds the folder containing `millionaire_client` to Python's import path. Change `PROJECT_DIR` after uploading this repository to Drive.

In [ ]:
# Mounted, your Drive must be. Visible to Python, the project folder becomes.
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_DIR = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment_api_client' if IN_COLAB else os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('millionaire_client exists:', os.path.isdir(os.path.join(PROJECT_DIR, 'millionaire_client')))

## 2. Optional dependency installation

Run this cell only when the model libraries are missing. The API smoke-test cells below do not need these packages.

In [ ]:
# Installed, model dependencies become, only when needed they are.
INSTALL_MODEL_DEPS = False

if INSTALL_MODEL_DEPS:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'accelerate',
        'sentencepiece',
        'torch'
    ])
else:
    print('Skipping install. Set INSTALL_MODEL_DEPS = True if imports fail later.')

## 3. API configuration

For quick experiments the password is present here. Before final submission/export, remove it or switch to a Colab Secret.

In [ ]:
# Here, the server and credentials live. Before submission, cleaned they should be.
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'MarianoAkaMery'
PASSWORD = 'Test1234!'

# Used instead, a Colab secret can be. Uncomment, safer you want it.
# from google.colab import userdata
# PASSWORD = userdata.get('poli-millionaire')

DEFAULT_COMPETITION_ID = 1
DEFAULT_MODE = 'text'

## 4. Official client import and login

This uses only the course-provided `millionaire_client` package.

In [ ]:
# Imported, the official client is. Invented endpoints, we use not.
from millionaire_client import MillionaireClient, AuthenticationError, GameError, MillionaireError

client = MillionaireClient(API_URL)

try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as {user.username} (role={user.role})')
except AuthenticationError as exc:
    print('Login failed:', exc)
    raise

## 5. List competitions

Use this before playing to choose the competition id. The ids are the public ids returned by the server.

In [ ]:
# Listed, available competitions are. Chosen, the competition id can be.
competitions = client.competitions.list_all()

for comp in competitions:
    print(f'{comp.id}: {comp.name} | max_levels={comp.max_levels} | questions={comp.question_count}')

## 6. Start one text game and inspect the question format

This cell starts a real session. It prints the exact fields that the bot will receive: question text and option ids/texts.

In [ ]:
# Started, one text game is. Observed, the API format becomes.
game = client.game.start(competition_id=DEFAULT_COMPETITION_ID, mode=DEFAULT_MODE)

print('Session ID:', game.session_id)
print('Mode:', game.mode)
print('Current level:', game.current_level)
print('Time remaining:', game.time_remaining)

question = game.current_question
print('\nQUESTION:', question.text)
print('\nOPTIONS:')
for opt in question.options:
    print(f'  id={opt.id} | text={opt.text}')

## 7. Manual answer test

Use this only when you want to verify that answering by `option_id` works. It submits a real answer to the current game.

In [ ]:
# Submitted, a manual answer can be. Real, this answer is.
RUN_MANUAL_ANSWER = False
MANUAL_OPTION_ID = None

if RUN_MANUAL_ANSWER:
    if MANUAL_OPTION_ID is None:
        raise ValueError('Set MANUAL_OPTION_ID to one of the printed option ids.')
    result = game.answer(MANUAL_OPTION_ID)
    print(result)
else:
    print('Manual answer skipped.')

## 8. Prompt builder

The model receives one question and the available options. It is asked to return only an option id, because the API requires an id.

In [ ]:
# Built, a compact prompt is. Only an id, the model should return.
def build_prompt(question):
    options_text = '\n'.join(f'{opt.id}. {opt.text}' for opt in question.options)
    return f"""You are playing a multiple-choice quiz.
Choose the single best answer.
Return only the numeric option id, with no explanation.

Question:
{question.text}

Options:
{options_text}

Answer id:"""

print(build_prompt(question))

## 9. Answer parsing

Local models sometimes return extra text. This parser extracts a valid option id and rejects ids that are not present in the current question.

In [ ]:
# Parsed, the model text is. Accepted, only real option ids are.
import re

def parse_option_id(raw_text, question):
    valid_ids = {opt.id for opt in question.options}
    numbers = [int(x) for x in re.findall(r'\d+', str(raw_text))]
    for number in numbers:
        if number in valid_ids:
            return number
    return None

print('Valid ids:', [opt.id for opt in question.options])

## 10. Local model wrapper

This wrapper loads an open-weight Hugging Face model locally inside Colab. The default models are small enough for initial experiments. You can replace them with Llama-family models if you have access and enough GPU memory.

In [ ]:
# Loaded locally, the model is. Called, no LLM API is.
import time

class LocalCausalLMAnswerer:
    def __init__(self, model_name, max_new_tokens=8):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.tokenizer = None
        self.model = None

    def load(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        if self.model is not None:
            return

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map='auto',
            torch_dtype='auto'
        )
        self.model.eval()

    def answer(self, question):
        self.load()
        prompt = build_prompt(question)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        start = time.time()
        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        elapsed = time.time() - start

        generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
        raw = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        option_id = parse_option_id(raw, question)

        return {
            'model': self.model_name,
            'raw': raw,
            'option_id': option_id,
            'elapsed_s': elapsed,
        }

## 11. Configure two local models

Start with small models to check the full pipeline. Later, replace `MODEL_A_NAME` or `MODEL_B_NAME` with stronger open-weight models that fit your Colab GPU.

In [ ]:
# Chosen, two small local models are. Bigger later, tried they can be.
MODEL_A_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_B_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

model_a = LocalCausalLMAnswerer(MODEL_A_NAME)
model_b = LocalCausalLMAnswerer(MODEL_B_NAME)

print('Model A:', MODEL_A_NAME)
print('Model B:', MODEL_B_NAME)

## 12. Test one model prediction without submitting

This checks model loading and output parsing on the current question. It does not call `game.answer`.

In [ ]:
# Predicted, one answer is. Submitted, it is not.
RUN_MODEL_DRY_RUN = False

if RUN_MODEL_DRY_RUN:
    prediction = model_a.answer(question)
    print(prediction)
else:
    print('Model dry run skipped. Set RUN_MODEL_DRY_RUN = True to load and test model A.')

## 13. Two-model ensemble

Both models answer the same question. If they agree, use that answer. If they disagree, prefer model B by default because it is larger. This rule is simple and easy to analyze.

In [ ]:
# Combined, two predictions are. Preferred, model B is when disagreement appears.
def ensemble_answer(question, first_model=model_a, second_model=model_b):
    pred_a = first_model.answer(question)
    pred_b = second_model.answer(question)

    if pred_a['option_id'] is not None and pred_a['option_id'] == pred_b['option_id']:
        chosen = pred_a['option_id']
        decision = 'agree'
    elif pred_b['option_id'] is not None:
        chosen = pred_b['option_id']
        decision = 'disagree_prefer_model_b'
    elif pred_a['option_id'] is not None:
        chosen = pred_a['option_id']
        decision = 'fallback_model_a'
    else:
        chosen = question.options[0].id
        decision = 'fallback_first_option'

    return {
        'chosen_option_id': chosen,
        'decision': decision,
        'pred_a': pred_a,
        'pred_b': pred_b,
    }

## 14. Play one full text game with logging

This cell plays a real game. To avoid accidental server load, it is disabled by default. Enable it only when the model dry run works.

In [ ]:
# Played, one full game can be. Logged, every step is.
import json
from datetime import datetime

RUN_FULL_GAME = False
MAX_LEVELS_TO_PLAY = 15
DELAY_BETWEEN_QUESTIONS_S = 1.0

def play_text_game_with_ensemble(competition_id=DEFAULT_COMPETITION_ID):
    logs = []
    active_game = client.game.start(competition_id=competition_id, mode='text')
    print('Started session:', active_game.session_id)

    while active_game.in_progress and active_game.current_level <= MAX_LEVELS_TO_PLAY:
        q = active_game.current_question
        if q is None:
            break

        print(f'\nLevel {active_game.current_level}')
        print(q.text)
        for opt in q.options:
            print(f'  {opt.id}. {opt.text}')

        before_answer_s = active_game.time_remaining
        decision = ensemble_answer(q)
        chosen_id = decision['chosen_option_id']
        print('Chosen:', chosen_id, '| decision:', decision['decision'])

        result = active_game.answer(chosen_id)
        print('Correct:', result.correct, '| timed_out:', result.timed_out, '| earned:', result.earned_amount)

        logs.append({
            'timestamp': datetime.utcnow().isoformat(),
            'session_id': active_game.session_id,
            'level': active_game.current_level,
            'question_id': q.id,
            'question_text': q.text,
            'options': [{'id': opt.id, 'text': opt.text} for opt in q.options],
            'time_remaining_before_answer_s': before_answer_s,
            'decision': decision,
            'result': {
                'correct': result.correct,
                'game_over': result.game_over,
                'timed_out': result.timed_out,
                'earned_amount': result.earned_amount,
                'current_level': result.current_level,
                'reached_level': result.reached_level,
            },
        })

        if result.game_over or result.timed_out:
            break

        time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    return active_game, logs

if RUN_FULL_GAME:
    final_game, game_logs = play_text_game_with_ensemble()
    print('\nFinal level:', final_game.current_level)
    print('Final earned amount:', final_game.earned_amount)
else:
    print('Full game skipped. Set RUN_FULL_GAME = True when ready.')

## 15. Save logs

Saving logs makes it possible to analyze model errors later and report benchmark results in the notebook.

In [ ]:
# Saved, experiment logs are. Studied later, mistakes can be.
SAVE_LOGS = False
LOG_DIR = os.path.join(PROJECT_DIR, 'runs')

if SAVE_LOGS:
    os.makedirs(LOG_DIR, exist_ok=True)
    output_path = os.path.join(LOG_DIR, f'run_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.json')
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(game_logs, f, indent=2, ensure_ascii=False)
    print('Saved logs to:', output_path)
else:
    print('Log saving skipped.')

## 16. Leaderboard check

After a real run, this cell shows the current leaderboard for the selected competition and mode.

In [ ]:
# Checked, leaderboard is. Compared, our score can be.
lb = client.leaderboard.get(competition_id=DEFAULT_COMPETITION_ID, limit=10, mode='text')

print('Leaderboard:', lb.competition.name)
for i, entry in enumerate(lb.entries, 1):
    marker = ' <-- us' if entry.username == USERNAME else ''
    print(f'{i}. {entry.username}: {entry.score} | level={entry.reached_level}{marker}')

## 17. Optional speech-mode notes

Speech mode is not the main strategy. If tested, the needed component is ASR: audio question/options become text, then the same text model answers. Text-to-speech is not needed because the API still accepts `option_id`.

In [ ]:
# Optional, speech mode is. Transcribed, audio must be before answering.
RUN_SPEECH_API_SHAPE_TEST = False

if RUN_SPEECH_API_SHAPE_TEST:
    speech_game = client.game.start(competition_id=DEFAULT_COMPETITION_ID, mode='speech')
    print('Speech session:', speech_game.session_id)
    question_audio = speech_game.fetch_audio_question()
    option_a_audio = speech_game.fetch_audio_option_next()
    print('Question audio bytes:', len(question_audio))
    print('Option A audio bytes:', len(option_a_audio))
else:
    print('Speech shape test skipped.')

## 18. Analysis checklist

Use this checklist to complete the final notebook narrative:

- Compare model A vs model B vs ensemble.
- Report average answer time and timeout risk.
- Report levels reached and earned amount across multiple runs.
- Discuss disagreement cases between models.
- Explain why RAG was not used in the first version.
- Optionally compare text mode with speech mode using ASR.